<a href='https://colab.research.google.com/github/Emelecto/QuantLab/blob/main/web/content/cursos/python/notebooks/c2_l2.ipynb' target='_parent'><img src='https://colab.research.google.com/assets/colab-badge.svg'/></a>

# C2-L2 · Limpieza, gaps y resample
Detecta huecos, compáralos y convierte datos horarios en velas diarias.

In [ ]:
import pandas as pd
from pathlib import Path

URL = 'https://raw.githubusercontent.com/Emelecto/QuantLab/main/web/content/cursos/python/data/c2_l2.csv'
try:
    df = pd.read_csv(URL, parse_dates=['timestamp'])
    print('Fuente: URL (Colab)')
except Exception as e:
    print('Sin red, uso fallback local:', e)
    for cand in [Path('../data/c2_l2.csv'), Path('data/c2_l2.csv'), Path('c2_l2.csv')]:
        if cand.exists():
            df = pd.read_csv(cand, parse_dates=['timestamp']); break
    print('Fuente: local')
print(df.shape)

In [ ]:
print('NaN por columna:')
print(df.isna().sum().to_string())
print('duplicados:', int(df.duplicated().sum()))
n_gaps = int(df['price'].isna().sum())
print(f'filas={len(df)} gaps={n_gaps}')

## Tratamiento: dropna vs ffill y su efecto en la volatilidad

In [ ]:
s_drop = df.dropna(subset=['price']).set_index('timestamp')['price']
s_ffill = df.set_index('timestamp')['price'].ffill()
vol_drop = s_drop.pct_change().std()
vol_ffill = s_ffill.pct_change().std()
print(f'vol (dropna)={vol_drop:.5f}  vol (ffill)={vol_ffill:.5f}')
print('ffill crea retornos 0% que achican la volatilidad: por eso se comparan.')

## Resample a velas diarias

In [ ]:
ts = df.dropna(subset=['price']).set_index('timestamp').sort_index()
velas = ts['price'].resample('D').ohlc()
vol = ts['volume'].resample('D').sum()
diario = velas.join(vol)
print(diario.round(2).to_string())
print(f'\ndías={len(diario)} horas_origen={len(ts)}')

In [ ]:
# Chequeo automático
assert n_gaps == 4, 'el CSV trae 4 gaps simulados'
assert len(diario) == 2, 'dos días de velas diarias'
assert (diario['volume'] > 0).all()
print('OK: limpieza y resample verificados')